# Notebook 05 — SqueezeNet 1.1 on ImageNet (pretrained)

Uses a **pretrained SqueezeNet 1.1** (1.24 M params, 58.2 % Top-1, no residual connections,
no FC layers) for BFT analysis. Weights download automatically from torchvision (~4.7 MB).

**Why SqueezeNet?**  
- Smallest pretrained ImageNet model with no residual connections  
- No large FC layers — the classifier is a single `Conv2d(512, 1000, 1)`, whose
  joint arbor matrix is only ~2 GB vs 75+ GB for AlexNet's first FC layer  

**Architecture note — squeeze spine:**  
SqueezeNet Fire modules have *parallel* expand branches (`expand1x1` + `expand3x3`).
Capturing all Conv2d layers would violate BFT's sequential assumption.
Instead, `collect_layer_dicts` is called with `layer_filter=squeezenet_spine_filter`,
which selects only the **squeeze-spine**: initial conv → 8 squeeze convs → classifier conv.
Each squeeze conv's `input_fmap` is the full concatenated output of the preceding Fire
module's expand branches — the true activation flowing through the network — so the
sequential chain is valid.

**Dataset:** Only the **validation set** is required (`ILSVRC2012_img_val.tar`, ~6.3 GB).
Set `IMAGENET_DIR` in §0 to a directory that contains a `val/` subfolder with the
50 000 validation images in ImageFolder layout (`val/<synset>/*.JPEG`).
No training images are needed.

## §0 — Imports & Config

In [1]:
%matplotlib inline
import sys, os, pickle, copy, warnings
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_distances, paired_cosine_distances
from torch.utils.data import DataLoader, Subset, TensorDataset

from src import (
    load_experiment, save_experiment,
    collect_layer_dicts,
    bft,
    build_scaffold_edges, scaffold_loading_from_edges, plot_scaffold_graph,
    extract_tree_nodes, plot_factor_tree,
    extract_factor_fingerprint, extract_fingerprint_matrix,
    compute_stimulus_similarity, project_stimuli_onto_tree,
    extract_factor_tree_nodes, compute_factor_activations,
    nodes_at_layer,
)

plt.rcParams.update({'figure.dpi': 80})
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGENET_DIR = '../data'   # ← SET THIS: root containing val/ (val-only, ~6.3 GB)

CACHE_ROOT = '../data/cache/nb05_imagenet_squeezenet'
FIG_DIR    = '../figs/05_imagenet_squeezenet'
os.makedirs(CACHE_ROOT, exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

# ── ImageNet constants ────────────────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
N_CLASSES     = 1000
IMG_SIZE      = 224

# ── Focus categories ──────────────────────────────────────────────────────────
# Each entry groups semantically related ImageNet class indices into one category.
# Indices sourced from HumanCategories (fl_classes.py / Geirhos et al. stimulus set).
N_SAMPLES_PER_CATEGORY = 50   # ← SET THIS: max samples per category for BFT

"""CATEGORY_CLASSES = {
    'airplane': [404, 895, 908],
    'ship':     [403, 472, 484, 510, 554, 576, 625, 628, 724, 780, 814, 871, 914],
    'car':      [407, 436, 468, 475, 479, 511, 581, 609, 627, 661, 751, 817],
    'bicycle':  [444, 671],
    'elephant': [101, 385, 386],
    'bear':     [294, 295, 296, 297],
    'dog':      [151, 152, 153, 154, 155, 156, 157, 158, 159, 160,
                 161, 162, 163, 164, 165, 166, 167, 168, 169, 170,
                 171, 172, 173, 174, 175, 176, 177, 178, 179, 180,
                 181, 182, 183, 184, 185, 186, 187, 188, 189, 190,
                 191, 192, 193, 194, 195, 196, 197, 198, 199, 200,
                 201, 202, 203, 204, 205, 206, 207, 208, 209, 210,
                 211, 212, 213, 214, 215, 216, 217, 218, 219, 220,
                 221, 222, 223, 224, 225, 226, 227, 228, 229, 230,
                 231, 232, 233, 234, 235, 236, 237, 238, 239, 240,
                 241, 242, 243, 244, 245, 246, 247, 248, 249, 250,
                 251, 252, 253, 254, 255, 256, 257, 258, 259, 260,
                 261, 262, 263, 264, 265, 266, 267, 268],
    'bird':     [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                 20, 21, 22, 23, 24, 80, 81, 82, 83, 84, 85, 86,
                 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98,
                 99, 100, 127, 128, 129, 130, 131, 132, 133, 134,
                 135, 136, 137, 138, 139, 140, 141, 142, 143, 144,
                 145, 146],
}"""

CATEGORY_CLASSES = {
    'airplane': [404, 895, 908],
    'ship':     [403, 472, 484],
    'car':      [407, 436, 468],
    'bicycle':  [444, 671],
    'elephant': [101, 385, 386],
    'bear':     [294, 295, 296],
    'dog':      [151, 152, 153],
    'bird':     [7, 8, 9],
}

CATEGORY_NAMES = list(CATEGORY_CLASSES.keys())
N_FOCUS        = len(CATEGORY_NAMES)   # 8

# Flat union of all class indices — used for data loading
ALL_FOCUS_IDX = sorted(set(i for idxs in CATEGORY_CLASSES.values() for i in idxs))

# Map: imagenet class index → category label (0–7)
IDX_TO_CAT = {idx: ci
              for ci, idxs in enumerate(CATEGORY_CLASSES.values())
              for idx in idxs}

# ── BFT hyperparameters — 10 squeeze-spine layers ────────────────────────────
# Layers: features.0 | squeeze×8 (Fire3–12) | classifier.1
K_MAX      = [4, 4, 4, 4, 4, 6, 6, 6, 6, N_FOCUS]
N_BRANCHES = [1, 1, 1, 1, 1, 1, 1, 1, 2, 5]
POOL_METHOD    = 'avg'
STIM_THRESHOLD = 0.0

# ── Ablation ──────────────────────────────────────────────────────────────────
ABLATION_FRACS = [0.02, 0.05, 0.10, 0.20, 0.30]
N_RANDOM_REPS  = 3

print('Config ready.')
print(f'Categories ({N_FOCUS}): {CATEGORY_NAMES}')
print(f'Total focus ImageNet classes: {len(ALL_FOCUS_IDX)}')

Config ready.
Categories (8): ['airplane', 'ship', 'car', 'bicycle', 'elephant', 'bear', 'dog', 'bird']
Total focus ImageNet classes: 23


## §1 — Model

**SqueezeNet 1.1** pretrained on ImageNet-1K. Downloads ~4.7 MB on first run.

In [3]:
# ── Data loaders ──────────────────────────────────────────────────────────────
normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
test_tfm  = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    normalize,
])

def _make_imagenet(split, transform):
    """Load ImageNet; falls back to ImageFolder if ILSVRC structure absent."""
    try:
        return torchvision.datasets.ImageNet(IMAGENET_DIR, split=split, transform=transform)
    except Exception:
        folder = 'train' if split == 'train' else 'val'
        return torchvision.datasets.ImageFolder(
            os.path.join(IMAGENET_DIR, folder), transform=transform)

val_ds   = _make_imagenet('val', test_tfm)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

CLASS_NAMES = val_ds.classes  # synset IDs, e.g. 'n01440764'
print(f'Val: {len(val_ds):,}   Classes: {N_CLASSES}')
print(f'Focus class synsets: {[CLASS_NAMES[c] for c in ALL_FOCUS_IDX]}')

Val: 50,000   Classes: 1000
Focus class synsets: ['n01514668', 'n01514859', 'n01518878', 'n01871265', 'n02085620', 'n02085782', 'n02085936', 'n02132136', 'n02133161', 'n02134084', 'n02504013', 'n02504458', 'n02687172', 'n02690373', 'n02701002', 'n02814533', 'n02835271', 'n02930766', 'n02951358', 'n02981792', 'n03792782', 'n04552348', 'n04592741']


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [4]:
# ── Focused val loader (focus categories only) ────────────────────────────────
val_targets  = np.array(val_ds.targets)
focus_idx    = np.where(np.isin(val_targets, ALL_FOCUS_IDX))[0]
focus_val_ds = Subset(val_ds, focus_idx)
focus_loader = DataLoader(focus_val_ds, batch_size=256, shuffle=False, num_workers=4)
print(f'Focus val samples: {len(focus_val_ds)}')

Focus val samples: 1150


In [5]:
# ── Load pretrained SqueezeNet 1.1 ────────────────────────────────────────────
model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'SqueezeNet 1.1 — parameters: {n_params:,}')

# Per-category top-1 accuracy on focus val samples
cat_correct = np.zeros(N_FOCUS)
cat_total   = np.zeros(N_FOCUS)
with torch.no_grad():
    for x, y in focus_loader:
        x = x.to(DEVICE)
        preds    = model(x).argmax(1).cpu().numpy()
        yt       = y.numpy()
        true_cats = np.array([IDX_TO_CAT.get(int(t), -1) for t in yt])
        pred_cats = np.array([IDX_TO_CAT.get(int(p), -1) for p in preds])
        for ci in range(N_FOCUS):
            mask = true_cats == ci
            cat_correct[ci] += (pred_cats[mask] == ci).sum()
            cat_total[ci]   += mask.sum()

cat_acc = cat_correct / (cat_total + 1e-12)
for ci, name in enumerate(CATEGORY_NAMES):
    print(f'  {name:10s}  Top-1: {cat_acc[ci]:.3f}  ({int(cat_correct[ci])}/{int(cat_total[ci])})')
print(f'\nMean category Top-1: {cat_acc.mean():.3f}')
print('(Full val set: Top-1 ≈ 0.582, Top-5 ≈ 0.806 per torchvision)')

SqueezeNet 1.1 — parameters: 1,235,496


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  airplane    Top-1: 0.847  (127/150)
  ship        Top-1: 0.687  (103/150)
  car         Top-1: 0.673  (101/150)
  bicycle     Top-1: 0.650  (65/100)
  elephant    Top-1: 0.833  (125/150)
  bear        Top-1: 0.747  (112/150)
  dog         Top-1: 0.567  (85/150)
  bird        Top-1: 0.833  (125/150)

Mean category Top-1: 0.730
(Full val set: Top-1 ≈ 0.582, Top-5 ≈ 0.806 per torchvision)


## §2 — BFT Factorization & Inspection

Layer data is collected via the **squeeze spine filter** — only the initial conv,
the 8 squeeze convs inside each Fire module, and the final classifier conv are captured.
This preserves BFT's sequential-layer assumption despite SqueezeNet's parallel expand branches.

In [6]:
def imdenorm(img, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    m = np.array(mean)[:, None, None]; s = np.array(std)[:, None, None]
    return np.clip((img * s + m).transpose(1, 2, 0), 0, 1)

def squeezenet_spine_filter(name, mod):
    """Select the sequential squeeze-spine: initial conv, squeeze convs, classifier conv."""
    return (name == 'features.0' or
            name == 'classifier.1' or
            (isinstance(mod, nn.Conv2d) and name.endswith('.squeeze')))

def filter_by_category(raw, n_per_category,
                        idx_to_cat=IDX_TO_CAT, n_categories=N_FOCUS):
    """Map ImageNet indices → category labels (0–N_FOCUS-1), sample n_per_category each.

    Samples are sorted by model confidence (highest first) within each category.
    Returns (filtered_dict, keep_indices).
    """
    orig_targets = raw['targets']
    cat_targets  = np.array([idx_to_cat.get(int(t), -1) for t in orig_targets])
    keep = []
    for ci in range(n_categories):
        ci_idx = np.where(cat_targets == ci)[0]
        if 'confidences' in raw and len(ci_idx):
            ci_idx = ci_idx[np.argsort(raw['confidences'][ci_idx])[::-1]]
        if len(ci_idx) > n_per_category:
            ci_idx = ci_idx[:n_per_category]
        keep.append(ci_idx)
    keep = np.sort(np.concatenate(keep))
    layer_data = [{**ld, 'input_fmap':  ld['input_fmap'][keep],
                         'output_fmap': ld['output_fmap'][keep]}
                  for ld in raw['layer_data']]
    result = {'images':       raw['images'][keep],
              'targets':      cat_targets[keep],      # category labels 0–7
              'orig_targets': orig_targets[keep],     # original ImageNet indices
              'layer_data':   layer_data}
    if 'confidences' in raw:
        result['confidences'] = raw['confidences'][keep]
    return result, keep

_p = os.path.join(CACHE_ROOT, 'raw_data_focus.pkl')
if os.path.exists(_p):
    with open(_p, 'rb') as f:
        raw0 = pickle.load(f)
    print('Loaded raw layer data from cache')
else:
    print('Collecting focus-category val data (squeeze spine only) …')
    raw0 = collect_layer_dicts(model, focus_loader, DEVICE,
                                only_correct=True,
                                layer_filter=squeezenet_spine_filter)
    with open(_p, 'wb') as f:
        pickle.dump(raw0, f)
    print('Saved.')

data0, _ = filter_by_category(raw0, N_SAMPLES_PER_CATEGORY)
all_images0   = data0['images']
all_targets0  = data0['targets']      # category labels 0–7
layer_inputs0 = [ld['input_fmap'] for ld in data0['layer_data']]
n_samples0    = len(all_images0)

print(f'{n_samples0} samples | {len(layer_inputs0)} spine layers')
print(f'Layer shapes: {[x.shape for x in layer_inputs0]}')
print(f'Layer names: {[ld["name"] for ld in data0["layer_data"]]}')
print(f'Category counts: {dict(zip(CATEGORY_NAMES, [int((all_targets0==i).sum()) for i in range(N_FOCUS)]))}')


Saved.
400 samples | 10 spine layers
Layer shapes: [(400, 3, 224, 224), (400, 64, 55, 55), (400, 128, 55, 55), (400, 128, 27, 27), (400, 256, 27, 27), (400, 256, 13, 13), (400, 384, 13, 13), (400, 384, 13, 13), (400, 512, 13, 13), (400, 512, 13, 13)]
Layer names: ['features.0', 'features.3.squeeze', 'features.4.squeeze', 'features.6.squeeze', 'features.7.squeeze', 'features.9.squeeze', 'features.10.squeeze', 'features.11.squeeze', 'features.12.squeeze', 'classifier.1']
Category counts: {'airplane': 50, 'ship': 50, 'car': 50, 'bicycle': 50, 'elephant': 50, 'bear': 50, 'dog': 50, 'bird': 50}


In [ ]:
# ── Run BFT ───────────────────────────────────────────────────────────────────

print('Running BFT …')
tree_root0 = bft(
    data0['layer_data'],
    k_max=K_MAX, n_branches=N_BRANCHES,
    conv_pool_method=POOL_METHOD,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity', verbose=1, n_jobs=3,
)

tree_nodes0   = extract_tree_nodes(tree_root0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)
l0_nodes0     = nodes_at_layer(tree_root0, 0)
K_root        = len(tree_root0['lambdas'])
print(f'Tree nodes: {len(tree_nodes0)}   K_root: {K_root}   L0 leaves: {len(l0_nodes0)}')

Running BFT …
[BFT] Layer 10/10 'classifier.1' (conv)  path=[]


In [ ]:
# ── Output layer factor panels ────────────────────────────────────────────────
fig, axes = plt.subplots(3, K_root, figsize=(K_root * 2, 6))
if K_root == 1: axes = axes.reshape(3, 1)

axes[0, 0].bar(range(K_root), tree_root0['lambdas'], color='steelblue')
axes[0, 0].set_title('λ values'); axes[0, 0].set_xlabel('factor')
for kk in range(1, K_root): axes[0, kk].axis('off')

for k in range(K_root):
    means = [tree_root0['img_factors'][:, k][all_targets0 == ci].mean()
             if (all_targets0 == ci).any() else 0.0
             for ci in range(N_FOCUS)]
    axes[1, k].bar(range(N_FOCUS), means, color=[f'C{i}' for i in range(N_FOCUS)], alpha=0.85)
    axes[1, k].set_xticks(range(N_FOCUS))
    axes[1, k].set_xticklabels([n[:6] for n in CATEGORY_NAMES], rotation=45, fontsize=7)
    axes[1, k].set_title(f'k={k} λ={tree_root0["lambdas"][k]:.2f}', fontsize=9)
    w = np.maximum(tree_root0['img_factors'][:, k], 0)
    w /= w.sum() + 1e-8
    avg = (all_images0 * w[:, None, None, None]).sum(0)
    axes[2, k].imshow(imdenorm(avg)); axes[2, k].axis('off')
    axes[2, k].set_title('avg img', fontsize=8)

plt.suptitle('Output layer factors — focus categories', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'output_layer_factors.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Conv kernel visualisation: top-8 channels per L0 leaf ────────────────────
N_SHOW = 8
for leaf in l0_nodes0[:3]:
    ld      = data0['layer_data'][leaf['layer_idx']]
    C_out, C_in, kH, kW = ld['weight'].shape
    neu_f   = leaf['neural_factors']
    top_f   = neu_f[:, 0].reshape(C_out, C_in, kH, kW)
    ch_load = top_f.reshape(C_out, -1).sum(1)
    top_ch  = np.argsort(ch_load)[::-1][:N_SHOW]

    if C_in == 3:  # initial conv: RGB kernels
        fig, axes = plt.subplots(1, len(top_ch), figsize=(2.2 * len(top_ch), 2.5))
        for col, ch in enumerate(top_ch):
            k = top_f[ch].transpose(1, 2, 0)
            k = (k - k.min()) / (k.max() - k.min() + 1e-8)
            axes[col].imshow(k); axes[col].axis('off')
            axes[col].set_title(f'ch{ch}\n{ch_load[ch]:.2f}', fontsize=8)
    else:  # squeeze convs: 1×1, show as bar chart per channel
        fig, axes = plt.subplots(1, len(top_ch), figsize=(2.2 * len(top_ch), 2.8))
        for col, ch in enumerate(top_ch):
            k    = top_f[ch].reshape(C_in, kH * kW).squeeze()
            vmax = np.abs(k).max()
            axes[col].bar(range(len(k)), k, color=['red' if v > 0 else 'blue' for v in k])
            axes[col].set_ylim(-vmax, vmax); axes[col].axis('off')
            axes[col].set_title(f'ch{ch}\n{ch_load[ch]:.2f}', fontsize=8)

    path_str = str(leaf['path'])
    fig.suptitle(f'{leaf["layer_name"]} ({leaf["layer_type"]}) path={path_str}: '
                 f'top-{N_SHOW} output channels', fontsize=10)
    plt.tight_layout()
    safe = path_str.replace(', ', '-').replace('[', '').replace(']', '')
    plt.savefig(os.path.join(FIG_DIR, f'kernels_{leaf["layer_name"]}_p{safe}.pdf'),
                bbox_inches='tight')
    plt.show()

In [ ]:
# ── Spatial activation maps (only for initial conv — it has spatial extent) ───
def get_spatial_activation_map(model, images_np, node, layer_data, device,
                                n_images=6, which='top'):
    scores  = node['img_factors'][:, 0]
    sel_idx = np.argsort(scores)[::-1][:n_images] if which == 'top' else np.argsort(scores)[:n_images]
    ld      = layer_data[node['layer_idx']]
    C_out, C_in, kH, kW = ld['weight'].shape
    neu_f  = node['neural_factors']
    ch_imp = np.maximum(neu_f[:, 0].reshape(C_out, C_in * kH * kW).sum(1), 0)
    if ch_imp.sum() > 0: ch_imp /= ch_imp.sum()

    fmaps = {}
    tmod  = dict(model.named_modules())[node['layer_name']]
    hook  = tmod.register_forward_hook(lambda m, i, o: fmaps.update({'out': o.detach().cpu()}))
    model.eval()
    with torch.no_grad():
        model(torch.from_numpy(images_np[sel_idx]).float().to(device))
    hook.remove()
    fmap    = fmaps['out'].numpy()
    spatial = np.maximum((fmap * ch_imp[None, :, None, None]).sum(1), 0)
    return spatial, sel_idx

# Only the initial conv (features.0) has spatial kernels; squeeze convs are 1×1
spatial_leaves = [n for n in l0_nodes0 if n['layer_name'] == 'features.0']
for leaf in spatial_leaves[:2]:
    top_maps, top_idx = get_spatial_activation_map(model, all_images0, leaf,
                                                    data0['layer_data'], DEVICE, 6, 'top')
    bot_maps, bot_idx = get_spatial_activation_map(model, all_images0, leaf,
                                                    data0['layer_data'], DEVICE, 6, 'bottom')
    n = len(top_idx)
    fig, axes = plt.subplots(4, n, figsize=(2.2 * n, 8))
    for col in range(n):
        axes[0, col].imshow(imdenorm(all_images0[top_idx[col]]))
        axes[0, col].set_title(CLASS_NAMES[all_targets0[top_idx[col]]][:8], fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(top_maps[col], cmap='hot'); axes[1, col].axis('off')
        axes[2, col].imshow(imdenorm(all_images0[bot_idx[col]]))
        axes[2, col].set_title(CLASS_NAMES[all_targets0[bot_idx[col]]][:8], fontsize=8)
        axes[2, col].axis('off')
        axes[3, col].imshow(bot_maps[col], cmap='hot'); axes[3, col].axis('off')
    for row, lbl in enumerate(['Top image', 'Top map', 'Bot image', 'Bot map']):
        axes[row, 0].set_ylabel(lbl)
    fig.suptitle(f'{leaf["layer_name"]} path={leaf["path"]}: spatial activation maps')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'spatial_{leaf["layer_name"]}.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# ── Class × path heatmap ──────────────────────────────────────────────────────
def get_all_paths(root):
    if not root['children']: return [[root]]
    return [[root] + sub for c in root['children'] for sub in get_all_paths(c)]

all_paths   = get_all_paths(tree_root0)
n_paths     = len(all_paths)
heatmap     = np.zeros((N_FOCUS, n_paths))
path_labels = []

for pi, path in enumerate(all_paths):
    leaf_node = path[-1]
    fi   = min(leaf_node['factor_idx'], leaf_node['img_factors'].shape[1] - 1)
    stim = leaf_node['img_factors'][:, fi]
    for ci in range(N_FOCUS):
        mask = all_targets0 == ci
        heatmap[ci, pi] = stim[mask].mean() if mask.any() else 0.0
    branch_seq  = leaf_node['path']
    path_labels.append('→'.join(str(b) for b in branch_seq) if branch_seq else 'main')

fig, ax = plt.subplots(figsize=(max(5, n_paths * 1.4), 4))
im = ax.imshow(heatmap, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_paths))
ax.set_xticklabels([f'P{i}\n({l})' for i, l in enumerate(path_labels)], fontsize=8)
ax.set_yticks(range(N_FOCUS))
ax.set_yticklabels(CATEGORY_NAMES)
ax.set_xlabel('Leaf path'); ax.set_ylabel('Category')
ax.set_title('Mean leaf stimulus weight per category × path')
plt.colorbar(im, ax=ax, label='Mean img_factor weight')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'class_path_heatmap.pdf'), bbox_inches='tight')
plt.show()

print('Dominant categories per path:')
for pi in range(n_paths):
    top3 = np.argsort(heatmap[:, pi])[::-1][:3]
    print(f'  P{pi} ({path_labels[pi]}): ' +
          ' | '.join(f'{CATEGORY_NAMES[i]} ({heatmap[i, pi]:.3f})' for i in top3))

In [ ]:
# ── Scaffold graph ────────────────────────────────────────────────────────────
# all_targets0 already contains category labels 0–(N_FOCUS-1)
scaffold_edges   = build_scaffold_edges(tree_root0)
scaffold_loading = scaffold_loading_from_edges(scaffold_edges, all_targets0, N_FOCUS)
fig, ax = plt.subplots(figsize=(14, 6))
plot_scaffold_graph(scaffold_edges, scaffold_loading, ax=ax)
ax.set_title('Scaffold graph — SqueezeNet 1.1 (node colour = dominant focus category)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'scaffold.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Top-N stimulus gallery per output factor ──────────────────────────────────
N_GAL = 8
fig, axes = plt.subplots(K_root, N_GAL, figsize=(N_GAL * 1.6, K_root * 1.8))
if K_root == 1: axes = axes.reshape(1, N_GAL)
for k in range(K_root):
    top_idx = np.argsort(tree_root0['img_factors'][:, k])[::-1][:N_GAL]
    for col, idx in enumerate(top_idx):
        axes[k, col].imshow(imdenorm(all_images0[idx]))
        axes[k, col].set_title(CATEGORY_NAMES[all_targets0[idx]], fontsize=7)
        axes[k, col].axis('off')
    axes[k, 0].set_ylabel(f'k={k} λ={tree_root0["lambdas"][k]:.2f}', fontsize=8,
                            rotation=0, labelpad=60, va='center')
plt.suptitle('Top stimuli per output factor (focus categories)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'top_stimuli_gallery.pdf'), bbox_inches='tight')
plt.show()

## §3 — Ablation (focus classes)

BFT vs magnitude vs random ablation sweep over the squeeze spine layers.
Weight lookup uses `ld['name'] + '.weight'` (robust for any layer filter).

In [ ]:
# ── Ablation helpers ──────────────────────────────────────────────────────────
def cnn_score_nmf(path_nodes):
    """NMF importance scores keyed by (layer_idx, i, j)."""
    scores = {}
    for idx, node in enumerate(path_nodes):
        l_idx = node['layer_idx']
        W     = node['W']
        fi    = path_nodes[idx + 1]['factor_idx'] if idx < len(path_nodes) - 1 else 0
        fi    = min(fi, node['neural_factors'].shape[1] - 1)
        flat  = np.abs(node['neural_factors'][:, fi])
        if node['layer_type'] == 'conv':
            C_out, C_in, kH, kW = W.shape
            imp = flat.reshape(C_out, C_in * kH * kW)
            for i in range(C_out):
                for j in range(C_in * kH * kW):
                    scores[(l_idx, i, j)] = float(imp[i, j])
        else:
            n_out, n_in = W.shape
            imp = flat.reshape(n_out, n_in)
            for i in range(n_out):
                for j in range(n_in):
                    scores[(l_idx, i, j)] = float(imp[i, j])
    return scores

def cnn_score_magnitude(layer_data):
    scores = {}
    for l_idx, ld in enumerate(layer_data):
        W    = ld['weight']
        flat = np.abs(W.reshape(W.shape[0], -1))
        for i in range(flat.shape[0]):
            for j in range(flat.shape[1]):
                scores[(l_idx, i, j)] = float(flat[i, j])
    return scores

def cnn_ablate_model(model, layer_data, scores, frac, method='top', seed=42):
    all_vals = np.array(list(scores.values()))
    if method == 'random':
        rng = np.random.RandomState(seed)
        keys = list(scores.keys())
        n_ablate = int(round(len(keys) * frac))
        ablate_keys = set(map(tuple, np.array(keys)[rng.permutation(len(keys))[:n_ablate]]))
    else:
        thr = np.quantile(all_vals, 1.0 - frac) if method == 'top' else np.quantile(all_vals, frac)
        ablate_keys = {k for k, v in scores.items()
                       if (v >= thr if method == 'top' else v <= thr)}

    abl   = copy.deepcopy(model)
    state = abl.state_dict()
    for l_idx, ld in enumerate(layer_data):
        pname = ld['name'] + '.weight'
        if pname not in state:
            continue
        W      = state[pname].clone()
        W_flat = W.reshape(W.shape[0], -1)
        for (li, i, j) in ablate_keys:
            if li == l_idx and i < W_flat.shape[0] and j < W_flat.shape[1]:
                W_flat[i, j] = 0.0
        state[pname] = W_flat.reshape(W.shape)
    abl.load_state_dict(state)
    return abl

def eval_focus_accuracy(model, loader, device,
                        idx_to_cat=IDX_TO_CAT, n_focus=N_FOCUS):
    """Per-category top-1: correct if predicted class falls in the same category."""
    model.eval()
    correct_arr = np.zeros(n_focus)
    total_arr   = np.zeros(n_focus)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            preds     = model(x).argmax(1).cpu().numpy()
            yt        = y.numpy()
            true_cats = np.array([idx_to_cat.get(int(t), -1) for t in yt])
            pred_cats = np.array([idx_to_cat.get(int(p), -1) for p in preds])
            for ci in range(n_focus):
                mask = true_cats == ci
                correct_arr[ci] += (pred_cats[mask] == ci).sum()
                total_arr[ci]   += mask.sum()
    return correct_arr / (total_arr + 1e-12)

print('Ablation helpers defined.')

In [ ]:
# ── §3.1: Comparative ablation sweep ─────────────────────────────────────────
METHODS = ['nmf_top', 'nmf_bottom', 'magnitude', 'random']
METHOD_STYLES = {
    'nmf_top':    dict(color='red',       lw=2.0, ls='-',  label='NMF (most important)'),
    'nmf_bottom': dict(color='orange',    lw=1.5, ls='--', label='NMF (least important)'),
    'magnitude':  dict(color='steelblue', lw=1.5, ls='-',  label='Magnitude'),
    'random':     dict(color='black',     lw=1.0, ls=':',  label='Random'),
}

abl_cache = os.path.join(CACHE_ROOT, 'ablation_focus.pkl')
if os.path.exists(abl_cache):
    with open(abl_cache, 'rb') as f:
        abl_results = pickle.load(f)
    print('Loaded ablation from cache')
else:
    mag_scores = cnn_score_magnitude(data0['layer_data'])

    def get_class_path_focus(root, targets, ci):
        mask = targets == ci
        if not root['children'] or not mask.any():
            node = root
            while node['children']: node = node['children'][0]
            return [root] if not root['children'] else [root, node]
        per_child = [root['children'][i]['img_factors'][mask, 0].mean()
                     for i in range(len(root['children']))]
        best = root['children'][int(np.argmax(per_child))]
        path = [root, best]; cur = best
        while cur['children']:
            cur = cur['children'][0]; path.append(cur)
        return path

    abl_results = {m: {} for m in METHODS}
    for ci, cat_name in enumerate(CATEGORY_NAMES):
        nmf_path = get_class_path_focus(tree_root0, all_targets0, ci)
        nmf_sc   = cnn_score_nmf(nmf_path)
        print(f'  {cat_name}', end=' ', flush=True)
        for method in METHODS:
            scores_m  = nmf_sc if method in ('nmf_top', 'nmf_bottom') else mag_scores
            direction = 'bottom' if method == 'nmf_bottom' else \
                        'random' if method == 'random' else 'top'
            abl_results[method][ci] = {}
            for frac in ABLATION_FRACS:
                reps = []
                for rep in range(N_RANDOM_REPS if method == 'random' else 1):
                    abl = cnn_ablate_model(model, data0['layer_data'], scores_m, frac,
                                            method=direction, seed=rep)
                    reps.append(eval_focus_accuracy(abl, focus_loader, DEVICE))
                abl_results[method][ci][frac] = np.mean(reps, axis=0)
        print('done')

    with open(abl_cache, 'wb') as f:
        pickle.dump(abl_results, f)
    print('Ablation done and cached.')

In [ ]:
# ── Ablation curves ───────────────────────────────────────────────────────────
cols = min(N_FOCUS, 4)
rows = (N_FOCUS + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.8), sharey=True)
axes_flat = np.array(axes).flatten()

for ci, cat_name in enumerate(CATEGORY_NAMES):
    ax = axes_flat[ci]
    for method in METHODS:
        accs = np.array([abl_results[method][ci][f][ci] for f in ABLATION_FRACS])
        st = METHOD_STYLES[method]
        ax.plot(ABLATION_FRACS, accs, color=st['color'], lw=st['lw'], ls=st['ls'],
                label=st['label'])
    ax.set_title(cat_name, fontsize=9)
    ax.set_xlabel('fraction ablated')
    if ci % cols == 0: ax.set_ylabel('accuracy')
    ax.set_ylim(0, 1.05)

axes_flat[-1].legend(loc='lower left', fontsize=8)
for ax in axes_flat[N_FOCUS:]: ax.axis('off')
plt.suptitle('Per-category accuracy vs fraction of squeeze-spine weights removed', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'ablation_curves.pdf'), bbox_inches='tight')
plt.show()

## §4 — Stimulus Analysis via NNLS

Round-trip consistency, ID val-split sanity check, and far-OOD analysis.

In [ ]:
# ── Round-trip test ───────────────────────────────────────────────────────────
N_RT   = min(200, n_samples0)
rng_rt = np.random.default_rng(0)
rt_sub = rng_rt.choice(n_samples0, N_RT, replace=False)
rt_inputs = [l[rt_sub] for l in layer_inputs0]

projected_rt = project_stimuli_onto_tree(tree_root0, rt_inputs)
F_orig_rt    = extract_fingerprint_matrix(tree_root0, rt_sub)
F_rt         = extract_fingerprint_matrix(projected_rt, np.arange(N_RT))
rt_sims      = 1.0 - paired_cosine_distances(F_orig_rt, F_rt)
rt_root      = 1.0 - paired_cosine_distances(
    tree_root0['img_factors'][rt_sub], projected_rt['img_factors'])

print(f'Round-trip (full):  mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}')
print(f'Round-trip (root):  mean={rt_root.mean():.4f}  std={rt_root.std():.4f}')
print()
print('Active-sample fractions per node (first 8):')
for tn in tree_nodes0[:8]:
    sw = tn['stimulus_weights_in']
    print(f'  layer={tn["layer_idx"]}  path={tn["path"]}  active={(sw > 0.01).mean():.3f}')

In [ ]:
# ── ID sanity check: val split-A vs split-B cross-similarity (4 focus categories) ─
# Randomly partition each category's val samples into two halves (A / B).
# Half-A uses the original tree projection; half-B is re-projected via NNLS.
# Strong within-category / weak between-category similarity validates the BFT encoding.
rng_id   = np.random.default_rng(42)
SHOW_CI  = list(range(4))   # first 4 categories: airplane, ship, car, bicycle
N_BLK    = 40
blocks   = {}

for ci in SHOW_CI:
    ci_idx = np.where(all_targets0 == ci)[0].copy()
    rng_id.shuffle(ci_idx)
    half  = len(ci_idx) // 2
    idx_A = ci_idx[:half]
    idx_B = ci_idx[half:]
    if len(idx_A) > N_BLK: idx_A = rng_id.choice(idx_A, N_BLK, replace=False)
    if len(idx_B) > N_BLK: idx_B = rng_id.choice(idx_B, N_BLK, replace=False)
    li_B   = [l[idx_B] for l in layer_inputs0]
    proj_B = project_stimuli_onto_tree(tree_root0, li_B)
    if len(idx_A):
        blocks[f'A-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(tree_root0, idx_A)
    if len(idx_B):
        blocks[f'B-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(proj_B, np.arange(len(idx_B)))

F_cross  = np.concatenate(list(blocks.values()), axis=0)
bl_sizes = [len(v) for v in blocks.values()]
bl_ends  = list(np.cumsum(bl_sizes))
S_cross  = compute_stimulus_similarity(F_cross)
centres  = np.array([0] + bl_ends[:-1]) + np.array(bl_sizes) / 2

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S_cross, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
for b in bl_ends[:-1]:
    ax.axhline(b - 0.5, color='k', lw=1.5); ax.axvline(b - 0.5, color='k', lw=1.5)
ax.set_xticks(centres); ax.set_xticklabels(list(blocks), rotation=45, ha='right', fontsize=8)
ax.set_yticks(centres); ax.set_yticklabels(list(blocks), fontsize=8)
ax.set_title('ID Sanity Check: val split-A vs split-B cross-similarity (4 focus categories)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'id_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Far-OOD: 4 synthetic 3-channel image types ────────────────────────────────
N_FAR = 200; rng_f = np.random.default_rng(99); C = 3
_mn   = np.array(IMAGENET_MEAN)[:, None, None]
_st   = np.array(IMAGENET_STD)[:, None, None]
_chk  = (np.indices((IMG_SIZE, IMG_SIZE)).sum(0) % 2)[None].astype(np.float32)

_noise_raw = np.clip(rng_f.normal(0.5, 0.25,
                     (N_FAR, C, IMG_SIZE, IMG_SIZE)).astype(np.float32), 0, 1)
_orig_px   = all_images0[:N_FAR] * _st + _mn
_inv_norm  = (np.clip(1.0 - _orig_px, 0, 1) - _mn) / _st

far_ood_arrays = {
    'gaussian_noise': ((_noise_raw - _mn) / _st).astype(np.float32),
    'uniform_gray':   np.zeros((N_FAR, C, IMG_SIZE, IMG_SIZE), dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR, C, IMG_SIZE, IMG_SIZE)).copy().astype(np.float32),
    'inverted_test':  _inv_norm.astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds = TensorDataset(torch.from_numpy(imgs), torch.zeros(len(imgs), dtype=torch.long))
    d  = collect_layer_dicts(model, DataLoader(ds, 128, shuffle=False), DEVICE,
                              only_correct=False,
                              layer_filter=squeezenet_spine_filter)
    li = [ld['input_fmap'] for ld in d['layer_data']]
    d['layer_inputs']   = li
    d['projected_root'] = project_stimuli_onto_tree(tree_root0, li)
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  n={len(imgs)}')

n_ex  = 6
fig, axes = plt.subplots(len(far_ood_arrays), n_ex,
                          figsize=(n_ex * 2, len(far_ood_arrays) * 2.2))
for row, (name, imgs) in enumerate(far_ood_arrays.items()):
    for col in range(n_ex):
        raw = np.clip(imgs[col] * _st + _mn, 0, 1).transpose(1, 2, 0)
        axes[row, col].imshow(raw); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9, rotation=30, ha='right', va='center')
plt.suptitle('Far OOD — example images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_examples.pdf'), bbox_inches='tight')
plt.show()

n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5))
for ax, (name, d) in zip(axes, far_ood_data.items()):
    all_idx = np.arange(len(d['images']))
    acts    = compute_factor_activations(d['factor_nodes'], all_idx)
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activations', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_trees.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Joint MDS: ID focus classes vs far-OOD ────────────────────────────────────
N_EACH = 60; rng_m = np.random.default_rng(7)
F_parts, li_last_parts, mds_labels, mds_markers = [], [], [], []

id_sub = rng_m.choice(n_samples0, min(N_EACH, n_samples0), replace=False)
_n = len(id_sub)
F_parts.append(extract_fingerprint_matrix(tree_root0, id_sub))
li_last_parts.append(layer_inputs0[-1][id_sub].reshape(_n, -1))
mds_labels  += ['ID-ImageNet'] * _n
mds_markers += ['o'] * _n

for (name, d), mkr in zip(far_ood_data.items(), ['^', 'D', 'P', 'X']):
    sub = rng_m.choice(len(d['images']), min(N_EACH, len(d['images'])), replace=False)
    _n  = len(sub)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], sub))
    li_last_parts.append(d['layer_inputs'][-1][sub].reshape(_n, -1))
    mds_labels  += [name] * _n
    mds_markers += [mkr]  * _n

F_joint    = np.concatenate(F_parts, axis=0)
coords_mds = MDS(n_components=2, dissimilarity='precomputed',
                  random_state=0, n_init=4).fit_transform(cosine_distances(F_joint))
pca_last   = PCA(n_components=2).fit_transform(np.vstack(li_last_parts))

unique_labels = list(dict.fromkeys(mds_labels))
cmap_tab     = plt.get_cmap('tab10')
label_color  = {l: cmap_tab(i % 10) for i, l in enumerate(unique_labels)}
label_marker = dict(zip(mds_labels, mds_markers))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for embed, ax, title in [
    (coords_mds, axes[0], 'MDS — factor fingerprint space'),
    (pca_last,   axes[1], 'PCA — last spine layer input'),
]:
    for i, lbl in enumerate(mds_labels):
        ax.scatter(embed[i, 0], embed[i, 1],
                   c=[label_color[lbl]], marker=label_marker[lbl], s=20, alpha=0.7)
    for lbl in unique_labels:
        ax.scatter([], [], c=[label_color[lbl]], marker=label_marker[lbl], s=50, label=lbl)
    ax.legend(markerscale=2, ncol=1, fontsize=8, title='stimulus type')
    ax.set(title=title, xlabel='Dim 1', ylabel='Dim 2')
plt.suptitle('ID ImageNet focus classes vs Far-OOD — MDS vs PCA', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'joint_mds.pdf'), bbox_inches='tight')
plt.show()

# Fingerprint intra vs inter-class similarity
F_all = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
S_all = compute_stimulus_similarity(F_all)
intra_vals, inter_vals = [], []
for fi in range(N_FOCUS):
    mask = all_targets0 == fi
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for fj in range(fi + 1, N_FOCUS):
        inter_vals.extend(S_all[np.ix_(mask, all_targets0 == fj)].ravel())

intra_arr = np.array(intra_vals); inter_arr = np.array(inter_vals)
print(f'Intra-class similarity: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class similarity: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(intra_arr, bins=60, alpha=0.6, label='Intra-class', density=True)
ax.hist(inter_arr, bins=60, alpha=0.6, label='Inter-class', density=True)
ax.axvline(intra_arr.mean(), color='C0', ls='--')
ax.axvline(inter_arr.mean(), color='C1', ls='--')
ax.set(xlabel='Cosine similarity', ylabel='Density',
       title='Factor fingerprint: intra vs inter-class similarity (focus classes)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show()